# M5 Forecasting Competition EDA

This notebook explores key ideas for understanding the M5 Forecasting Competition data, which will help in crafting a Chronos-2 zero-shot baseline.

# Imports

In [1]:
import sys
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")

import os
import pandas as pd
import matplotlib.pyplot as plt
from m5_benchmarks import M5BenchmarkSuite
from m5_dataprep import M5DataPipeline
from m5_evaluator import trim_series_to_active, M5Evaluator
import torch

# Config

# Load and format the sales data

# Utility functions

In [2]:
# Tags to isolate experiments
dataprep_tags = "baseline" # data tags for data preparation choices
modeling_tags = "baseline" # modeling tags for modeling
postproc_tags = "baseline" # postprocessing tags for postprocessing

cutoff_day = "2016-05-22" #

new_cutoff = pd.to_datetime(cutoff_day) - pd.Timedelta(days=28)
cutoff_day = new_cutoff.strftime('%Y-%m-%d')
print(f"Cutoff day: {cutoff_day}")

dataprep_config = {
    "tag": "default",
    "path": "",
    "hist_cols": [],
    "fut_cols": [],  # for synamic covariates among ["wm_yr_wk", "wday", "month", "year", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI", "sell_price"]
    "stat_cols": ["item_id", "dept_id", "cat_id", "store_id", "state_id"], # for static covariates among ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
}

# config for autogluon wrapper
wrapper_dict = {
    "cutoff_day": cutoff_day,
    "level": 12,
    "data_tag": "default",
    "target": "sales_quantity",
    "eval_metric": "RMSSE",
    "enable_ensemble": False,
    "skip_model_selection": True,
    "verbosity": 0
}

Cutoff day: 2016-04-24


In [3]:

# CALL THE CACHE WRAPPER, NOT PREPARE DIRECTLY

# Initialize
pipeline = M5DataPipeline(config=dataprep_config)
path = "/mnt/lab/datasets/M5/jointed_M5.parquet"
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = pipeline.get_prepared_data(path, cutoff_day, level=12) #, force_reprepare=True)



--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/default/level_12/20160424 ---


## Actual data for evaluations

In [9]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    if "id" not in gt_wide.columns:
        gt_wide["id"] = gt_wide["item_id"] + "_" + gt_wide["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide.columns if c.startswith("d_")]
    long = gt_wide.melt(id_vars=["id"], value_vars=day_cols,
                        var_name="d", value_name="sales_quantity")
    cal = pd.read_csv(calendar_path, usecols=["d", "date"])
    cal["date"] = pd.to_datetime(cal["date"])
    long = long.merge(cal, on="d", how="left").drop(columns=["d"])
    long["id"] = long["id"].str.replace("_evaluation", "", regex=False)
    return long[["id", "date", "sales_quantity"]]

DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

DATA_TAG = "sales_only"

if cutoff_day == "2016-05-22":
    _eval_raw = pd.read_csv(ACTUALS_PATH)
    df_actual = _wide_to_long(_eval_raw, CALENDAR_PATH)
    del _eval_raw
else:
    df_actual = (
        pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
        .rename(columns={"sold": "sales_quantity"})
    )
    df_actual["id"] = (
        df_actual["id"].astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation",  "", regex=False)
    )

print(f"Actuals : {df_actual.shape}  |  "
      f"{df_actual['date'].min().date()} \u2192 {df_actual['date'].max().date()}")

Actuals : (59181090, 3)  |  2011-01-29 → 2016-05-22


# Modeling

In [5]:
# create the benchmark suite
suite = M5BenchmarkSuite(horizon=28, model_path ="/mnt/lab/nmwamsojo/autogluon_models/")

# Run a single method
method = "ES_bu"
fcst_df = suite.run(train_df=hist_df_trimmed, methods=[method], wrapper_dict=wrapper_dict, force_refit=False)[method]

--- Running ES_bu via AutoGluon (statsforecast backend) ---


Renaming existing column 'item_id' -> '__item_id' to avoid name collisions.


# Evaluation

In [13]:
# create evaluator with trimmed data
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)
# compute metrics
metrics1 = evaluator.evaluate_all(fcst_df, df_actual)

print(f"ES WRMSSE: {metrics1['WRMSSE']:.4f} | WAPE: {metrics1['WAPE_L12']:.2%}")

  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
ES WRMSSE: 0.7570 | WAPE: 72.00%
